<img src='https://github.com/Deci-AI/super-gradients/blob/master/docs/assets/SG_img/SG%20-%20Horizontal%20Glow%202.png?raw=true'>

## If you want to learn more about the EfficientNet family of model architectures, be sure to check out [my FREE course on Udemy](https://www.udemy.com/course/supergradients-efficientnet).

In this notebook, you'll use the SuperGradients training library to perform classifcation of yoga poses.

[SuperGradients](https://github.com/Deci-AI/super-gradients) is an open-source PyTorch based training library that has a number of pre-trained models for you to use, training recipies that will get you amazing accuracy, and many [training tricks](https://www.deeplearningdaily.community/t/tips-for-training-your-neural-networks/307) that you can use with just the "flip of a switch". For this example you'll use an EfficientNetB0 to perfom the classification. You can check out our [model zoo](https://github.com/Deci-AI/super-gradients/blob/master/src/super_gradients/training/Computer_Vision_Models_Pretrained_Checkpoints.md) and use any of the pretrained models we have available.

Feel free to reach out to me on my community forum, [Deep Learning Daily (free and open to all)](https://www.deeplearningdaily.community/), should you have any questions.

In [ ]:
%%capture
# Be sure to restart the kernal after installation
!pip install imutils
!pip install super-gradients==3.0.7
!pip install torchinfo

In [ ]:
from pathlib import Path, PurePath
import os
from sklearn.model_selection import train_test_split
from imutils import paths
import shutil
import random
import matplotlib.pyplot as plt
from PIL import Image
import math
from torchvision import transforms
import numpy as np
from torchvision import datasets
from torch.utils.data import DataLoader
from super_gradients.training import models
from super_gradients.training import dataloaders
from super_gradients.training import Trainer
from super_gradients.training import training_hyperparams
from super_gradients.training.metrics.classification_metrics import Accuracy, Top5
from super_gradients.training.utils.early_stopping import EarlyStop
from super_gradients.training.utils.callbacks import Phase
import torch
import textwrap
from typing import List, Tuple
import torchvision
import pathlib
import torchinfo

# Config file to hold some variables

In [ ]:
class config:

    # specify the paths to datasets
    DOWNLOAD_DIR = '/kaggle/input/yoga-pose-classification/YogaPoses'
    TRAIN_DIR = 'data/train'
    VAL_DIR = 'data/val'
    TEST_DIR = 'data/test'

    # set the input height and width
    INPUT_HEIGHT = 224
    INPUT_WIDTH = 224

    # set the input height and width
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Split images into train, validation, and test sets

In [ ]:
def split_image_folder(image_paths:str,
                folder:str):
  """
  This function creates the destination folder if it doesn't exist,
  loops over the image paths, extracts image name and label from the path,
  creates the label folder if it doesn't exist, makes the destination image 
  path and copies the current image to it.

  Parameters
  ----------
  image_paths : str
    Where the image is located
  folder : str
    train/validation path 

  """
  data_path = Path(folder)

  if not data_path.is_dir():
    data_path.mkdir(parents=True, exist_ok=True)

  for path in image_paths:
    full_path = Path(path)
    image_name = full_path.name
    label = full_path.parent.name
    label_folder = data_path / label

    if not label_folder.is_dir():
        label_folder.mkdir(parents=True, exist_ok=True)

    destination = label_folder / image_name
    shutil.copy(path, destination)

In [ ]:
from sklearn.model_selection import train_test_split

# load all the image paths and split them into train & validation sets
print("[INFO] Getting file paths and shuffling")
image_paths = list(sorted(paths.list_images(config.DOWNLOAD_DIR)))

print("[INFO] Configuring training and testing data")
class_names = [Path(x).parent.name for x in image_paths]
train_paths, rest_of_paths = train_test_split(image_paths, stratify=class_names, test_size=0.15, shuffle=True, random_state=42)

class_names_ = [Path(x).parent.name for x in rest_of_paths]
val_paths, test_paths = train_test_split(rest_of_paths, stratify=class_names_, test_size=0.50, shuffle=True, random_state=42)


# copy the training and validation images to directories
print("[INFO] Creating ImageFolder's for training and validation datasets")
split_image_folder(train_paths, config.TRAIN_DIR)
split_image_folder(val_paths, config.VAL_DIR)
split_image_folder(val_paths, config.TEST_DIR)

# Plot some images

In [ ]:
train_image_path_list = list(Path(config.TRAIN_DIR).glob("*/*.jpg"))
train_image_path_sample = random.sample(population=train_image_path_list, k=20)

def examine_images(images:list):
    num_images = len(images)
    num_rows = int(math.ceil(num_images/5))
    num_cols = 5
    
    fig, axs = plt.subplots(num_rows, num_cols, figsize=(30, 30),tight_layout=True)
    axs = axs.ravel()

    for i, image_path in enumerate(images[:num_images]):
        image = Image.open(image_path)
        label = PurePath(image_path).parent.name
        axs[i].imshow(image)
        axs[i].set_title(f"Pose: {label}", fontsize=40)
        axs[i].axis('off')
    plt.show()

examine_images(train_image_path_sample)

## I'm curious about the distribution of classes

In [ ]:
path = Path(config.TRAIN_DIR)

# Get a list of all subdirectories in the root directory
subdirs = [d for d in path.iterdir() if d.is_dir()]

# Initialize a dictionary to store the count of images in each subdirectory
image_count = {}

# Iterate through each subdirectory
for subdir in subdirs:
    subdir_images = list(sorted(paths.list_images(subdir)))
    image_count[subdir.name] = len(subdir_images)

plt.bar(image_count.keys(), image_count.values())
# add the count numbers on top of the bars
for i, (subdir, count) in enumerate(image_count.items()):
    plt.text(i, count + 3, str(count), ha='center')

# set the title and labels for the plot
plt.title("Number of Images in Each Subdirectory")
plt.xlabel("Subdirectories")
plt.ylabel("Counts")

# show the plot
plt.show()

# Initialize Augmentations

In this case it would make sense to perfom spatial augmentations, such as:

* Rotations
* Flips
* Transposes
* Shear
* Non-uniform scaling
* Crops


You can use torchvision to instantiate the transforms. 

In [ ]:
# initialize our data augmentation functions
resize = transforms.Resize(size=(config.INPUT_HEIGHT,config.INPUT_WIDTH))
make_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=config.IMAGENET_MEAN, std=config.IMAGENET_STD)
center_cropper = transforms.CenterCrop((config.INPUT_HEIGHT,config.INPUT_WIDTH))
random_horizontal_flip = transforms.RandomHorizontalFlip(p=0.5)
random_vertical_flip = transforms.RandomVerticalFlip(p=0.5)
random_rotation = transforms.RandomRotation(degrees=180)
random_crop = transforms.RandomCrop(size=(config.INPUT_HEIGHT,config.INPUT_WIDTH))
random_erasing = transforms.RandomErasing()
# randomly_choose_one 

# initialize our training and validation set data augmentation pipeline
train_transforms = transforms.Compose([
  resize, 
  center_cropper,
  random_crop,
  random_horizontal_flip,
  random_vertical_flip,
  random_rotation,
  make_tensor,
  normalize
])

val_transforms = transforms.Compose([resize, make_tensor, normalize])

# Examine what a single image looks like after the augmentation pipeline

In [ ]:
img = Image.open(train_image_path_sample[0])

plt.imshow(img)
plt.axis("off")
plt.title("Original Image")
plt.show()

img_tensor = train_transforms(img)
img_tensor = img_tensor.numpy().transpose((1, 2, 0))
img_tensor = np.clip(img_tensor, 0, 1)

plt.imshow(img_tensor)
plt.axis("off")
plt.title("Augmented Image")
plt.show()

# Instantiate DataLoaders

In [ ]:
def create_dataloaders(
    train_dir: str, 
    val_dir: str,
    test_dir: str,
    train_transform: transforms.Compose,
    val_transform:  transforms.Compose,
    test_transform:  transforms.Compose,
    batch_size: int, 
    num_workers: int=2
):
  """Creates training and validation DataLoaders.
  Args:
    train_dir: Path to training data.
    val_dir: Path to validation data.
    transform: Transformation pipeline.
    batch_size: Number of samples per batch in each of the DataLoaders.
    num_workers: An integer for number of workers per DataLoader.
  Returns:
    A tuple of (train_dataloader, val_dataloader, class_names).
  """
  # Use ImageFolder to create dataset
  train_data = datasets.ImageFolder(train_dir, transform=train_transform)
  val_data = datasets.ImageFolder(val_dir, transform=val_transform)
  test_data = datasets.ImageFolder(test_dir, transform=val_transform)  

  print(f"[INFO] training dataset contains {len(train_data)} samples...")
  print(f"[INFO] validation dataset contains {len(val_data)} samples...")
  print(f"[INFO] test dataset contains {len(test_data)} samples...")

  # Get class names
  class_names = train_data.classes
  print(f"[INFO] dataset contains {len(class_names)} labels...")

  # Turn images into data loaders
  print("[INFO] creating training and validation set dataloaders...")
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      drop_last=True,
      num_workers=num_workers,
      pin_memory=True,
      persistent_workers=True
  )
  val_dataloader = DataLoader(
      val_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
      drop_last=False,
      persistent_workers=True
  )

  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
      drop_last=False,
      persistent_workers=True
  )

  return train_dataloader, val_dataloader, test_dataloader, class_names

In [ ]:
train_dataloader, valid_dataloader, test_dataloader, class_names = create_dataloaders(train_dir=config.TRAIN_DIR,
                                                                     val_dir=config.VAL_DIR,
                                                                     test_dir=config.TEST_DIR,
                                                                     train_transform=train_transforms,
                                                                     val_transform=val_transforms,
                                                                     test_transform=val_transforms,
                                                                     batch_size=16)

NUM_CLASSES = len(class_names)

# Get the training recipe and instantiate the SuperGradients trainer

First thing we do is get the training recipe. Learn more about what a training recipe is and how to use it [here](https://github.com/Deci-AI/super-gradients/blob/master/tutorials/what_are_recipes_and_how_to_use.ipynb).

In [ ]:
efficientnet_training_params =  training_hyperparams.get('training_hyperparams/imagenet_efficientnet_train_params')

In [ ]:
efficientnet_training_params

The training recipe we have offer a solid starting point for you to futher tune your hyperparameters. They were found by our team at Deci and perfom well on the ImageNet dataset.

Let's override some of the values here, first we'll start by enabling early stopping.

In [ ]:
# To reduce clutter in the notebook I've turned the verbosity off, you can turn it on to see the full output
early_stop_acc = EarlyStop(Phase.VALIDATION_EPOCH_END, monitor="Accuracy", mode="max", patience=20, verbose=False)
early_stop_val_loss = EarlyStop(Phase.VALIDATION_EPOCH_END, monitor="LabelSmoothingCrossEntropyLoss", mode="min", patience=20, verbose=False)

efficientnet_training_params["train_metrics_list"] = [Accuracy()]
efficientnet_training_params["valid_metrics_list"] = [Accuracy()]
efficientnet_training_params["phase_callbacks"] = [early_stop_acc, early_stop_val_loss]

# Set the silent mode to True to reduce clutter in the notebook, you can turn it on to see the full output
efficientnet_training_params["silent_mode"] = True
# We'll turn off the use of exponential moving average and zero weight decay on bias and batch norm
efficientnet_training_params['ema'] = False
efficientnet_training_params['zero_weight_decay_on_bias_and_bn'] = False

As an FYI, you can pass in a different optimizer if you'd like. Learn more about optimizers [here](https://github.com/Deci-AI/super-gradients/blob/cd8cb385ffdaec84df917e3da5a89defcedf451a/documentation/source/optimizers.md).

The last thing we'll do here is enable label smoothing cross entropy, set the maximum number of epochs to 200, and lower the learning rate to 0.0001.


In [ ]:
efficientnet_training_params["criterion_params"] = {'smooth_eps': 0.25}
efficientnet_training_params["max_epochs"] = 200
efficientnet_training_params["initial_lr"] = 0.0001

# Training

You're looking to classify your pose into one of 5 poses.

That means a random classifier has a 1 in 5, or 20%, chance of correctly classifying the pose. This is the dummy baseline that you want to beat.

You'll fine-tune the pre-trained EfficientNetB0 model on the Yoga Pose dataset. By fine-tuning, you're updating the parameters of a pre-trained model to make it better suited for a specific task. This is typically done by unfreezing some or all of the layers of the pre-trained model and training them on the new task data. 

Fine-tuning allows the model to leverage the knowledge it has learned from the pre-training task and improve its performance on the new task. 

## Fine-tuning EfficientNetB0 
As soon as we pass the value for `num_classes` to `models.get` SuperGradients will replace the classification head for us.

In [ ]:
efficientnet_full_model = models.get(model_name='efficientnet_b0', num_classes= len(class_names), pretrained_weights='imagenet')
full_model_trainer = Trainer(experiment_name="0_Baseline_Experiment", ckpt_root_dir='checkpoints')

full_model_trainer.train(model=efficientnet_full_model, 
              training_params=efficientnet_training_params, 
              train_loader=train_dataloader,
              valid_loader=valid_dataloader)

The training was able to get some pretty decent accuracy, I'm actually kinda suprised. 

But, let's see how well it performs on the test set. Fist, you'll need to grab the best model. 

You can do that as follows:

In [ ]:
# Load the best model that we trained
best_full_model = models.get('efficientnet_b0',
                        num_classes=len(class_names),
                        checkpoint_path=os.path.join(full_model_trainer.checkpoints_dir_path, "ckpt_best.pth"))

To test on the test data, run the following code. It will show you the loss metric and the accuracy metric. 

In [ ]:
full_model_trainer.test(model=best_full_model,
            test_loader=test_dataloader,
            test_metrics_list=['Accuracy'])

## Plot predictions

Now we can plot the predictions

In [ ]:
import requests
def pred_and_plot_image(image_path: str, 
                        subplot: Tuple[int, int, int],  # subplot tuple for `subplot()` function
                        class_names: List[str] = class_names,
                        model: torch.nn.Module = best_full_model,
                        image_size: Tuple[int, int] = (config.INPUT_HEIGHT, config.INPUT_WIDTH),
                        transform: torchvision.transforms = None,
                        device: torch.device=config.DEVICE):

    if isinstance(image_path, pathlib.PosixPath):
        img = Image.open(image_path)
    else: 
        img = Image.open(requests.get(image_path, stream=True).raw)

    # create transformation for image (if one doesn't exist)
    if transform is None:
        transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=config.IMAGENET_MEAN,
                                 std=config.IMAGENET_STD),
        ])
    transformed_image = transform(img)

    # make sure the model is on the target device
    model.to(device)

    # turn on model evaluation mode and inference mode
    model.eval()
    with torch.inference_mode():
        # add an extra dimension to image (model requires samples in [batch_size, color_channels, height, width])
        transformed_image = transformed_image.unsqueeze(dim=0)

        # make a prediction on image with an extra dimension and send it to the target device
        target_image_pred = model(transformed_image.to(device))

    # convert logits -> prediction probabilities (using torch.softmax() for multi-class classification)
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

    # convert prediction probabilities -> prediction labels
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    # actual label
    ground_truth = PurePath(image_path).parent.name

    # plot image with predicted label and probability 
    plt.subplot(*subplot)
    plt.imshow(img)
    if isinstance(image_path, pathlib.PosixPath):
        title = f"Ground Truth: {ground_truth} | Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}"
    else:
        title = f"Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}"
    plt.title("\n".join(textwrap.wrap(title, width=20)))  # wrap text using textwrap.wrap() function
    plt.axis(False)
    

def plot_random_test_images(model):
    num_images_to_plot = 30
    test_image_path_list = list(Path(config.TEST_DIR).glob("*/*.jpg")) # get list all image paths from test data 
    test_image_path_sample = random.sample(population=test_image_path_list, # go through all of the test image paths
                                           k=num_images_to_plot) # randomly select 'k' image paths to pred and plot

    # set up subplots
    num_rows = int(np.ceil(num_images_to_plot / 5))
    fig, ax = plt.subplots(num_rows, 5, figsize=(15, num_rows * 3))
    ax = ax.flatten()

    # Make predictions on and plot the images
    for i, image_path in enumerate(test_image_path_sample):
        pred_and_plot_image(model=model, 
                            image_path=image_path,
                            class_names=class_names,
                            subplot=(num_rows, 5, i+1),  # subplot tuple for `subplot()` function
                            image_size=(config.INPUT_HEIGHT, config.INPUT_WIDTH))

    # adjust spacing between subplots
    plt.subplots_adjust(wspace=1)
    plt.show()

In [ ]:
plot_random_test_images(best_full_model)

In [ ]:
pred_and_plot_image(image_path='https://www.yogajournal.com/wp-content/uploads/2021/07/Warrior-I-Pose-Virabhadrasana-II-1.jpg', subplot=(1, 1, 1))

In [ ]:
pred_and_plot_image(image_path='https://images.ctfassets.net/3s5io6mnxfqz/5p1o5aDy0gthIFdGquvAzM/e70b350d1f675ce8bb2e0a6e1aa5f576/AdobeStock_360772900.jpeg', subplot=(1, 1, 1))

In [ ]:
pred_and_plot_image(image_path='https://beextrayoga.com/wp-content/uploads/2019/11/Nicole_202-Goddess-Pose-scaled.jpg', subplot=(1, 1, 1))

In [ ]:
pred_and_plot_image(image_path='https://yogapractice.com/wp-content/uploads/2019/05/How-to-Do-Tree-Pose-1.jpg', subplot=(1, 1, 1))

# Your homework

Copy/fork this notebook and try some different architectures.

If you have a question you can leave a comment on this notebook, or visit the community and post it in the [Q&A section](https://www.deeplearningdaily.community/c/qanda/8).

## Use a different pretrained model

You can change the model you use. Take a look at the [SG model zoo](https://github.com/Deci-AI/super-gradients/blob/master/src/super_gradients/training/Computer_Vision_Models_Pretrained_Checkpoints.md)

For example, if you wanted to use RegNet you would do the following:

```
resnet_imagenet_model = models.get(model_name='regnetY800', num_classes=NUM_CLASSES, pretrained_weights='imagenet)
resnet_params =  training_hyperparams.get('training_hyperparams/imagenet_regnetY_train_params')
```

Note you can also pass 'model_name=regnetY200', 'model_name=regnetY400', 'model_name=regnetY600' to try a variety of the architecture

For ResNet50, you would do:

```
resnet_imagenet_model = models.get(model_name='resnet50', num_classes=NUM_CLASSES, pretrained_weights='imagenet)
resnet_params =  training_hyperparams.get('training_hyperparams/imagenet_resnet50_train_params')
```


Note you can also pass 'model_name=resnet18' or 'model_name=resnet34' to try a variety of the architecture

For MobileNetV2, you would do:

```
mobilenet_imagenet_model = models.get(model_name='mobilenet_v2', num_classes=NUM_CLASSES, pretrained_weights='imagenet)
resnet_params =  training_hyperparams.get('training_hyperparams/imagenet_mobilenetv2_train_params')
```

For MobileNetV3, you would do:

```
mobilenet_imagenet_model = models.get(model_name='mobilenet_v3_large', num_classes=NUM_CLASSES, pretrained_weights='imagenet)
resnet_params =  training_hyperparams.get('training_hyperparams/imagenet_mobilenetv3_train_params')
```

Note you can also pass 'model_name=mobilenet_v3_small' to try a variety of the architecture


For ViT, you would do:


```
vit_imagenet_model = models.get(model_name='vit_base', num_classes=NUM_CLASSES, pretrained_weights='imagenet')
vit_params =  training_hyperparams.get("training_hyperparams/imagenet_vit_train_params")
```

Note you can also pass 'model_name=vit_large' to try a variety of the architecture


I encourage you play around with different optimizers, all you have to do is change the value of `training_params["optimizer"]`. You can use one of ['Adam','SGD','RMSProp'] out of the box. You can play around with the optimizer params as well.

In general, play and tweak around the training recipies...

## Training recipes

SuperGradients has a number of [training recipes](https://github.com/Deci-AI/super-gradients/tree/master/src/super_gradients/recipes) you can use. [See here](https://github.com/Deci-AI/super-gradients/blob/master/src/super_gradients/recipes/training_hyperparams/default_train_params.yaml) for more information about the training params.

If you're using Weights and Biases to track your experiments, you would do the following

```
sg_logger: wandb_sg_logger
sg_logger_params:
project_name: <YOUR PROJECT NAME>
entity: algo
api_server: https://wandb.research.deci.ai
save_checkpoints_remote: True
save_tensorboard_remote: True
save_logs_remote: True
```

### Instantiate the training params